# **Deep Learning Avanzado (INF3813) - MIA**. <br> Práctico Transferencia de conocimiento: Feature Extraction y Fine-tuning
---
---

**Profesor:**
- Felipe del Río

**Ayudante:**
- Andreina Cota
---
---

# **Agenda**

En este práctico veremos cómo aprovechar un modelo preentrenado en ImageNet para clasificar especies de flores con pocos datos etiquetados. Para eso entrenaremos y compararemos tres modelos con la misma arquitectura (ResNet18):

1. **Preparación:** descargamos el dataset *flowers*, lo exploramos y armamos el entorno de entrenamiento (hiperparámetros, `DataLoader`s, transformaciones y funciones de apoyo).
2. **Modelo base:** entrenamos una ResNet18 desde cero, que nos servirá como punto de comparación.
3. **Feature Extraction:** usamos la ResNet18 preentrenada como extractor de características fijo: congelamos sus pesos y solo entrenamos la última capa.
4. **Fine-tuning:** partimos de los mismos pesos preentrenados, pero ahora ajustamos toda la red a nuestro problema.
5. **Resultados:** comparamos los tres modelos en accuracy y loss sobre los sets de entrenamiento, validación y test.
6. **¿Por qué funcionan las características?:** visualizamos el espacio de embeddings de una red con pesos aleatorios y de una preentrenada, y usamos esos embeddings para hacer búsqueda semántica de imágenes.

A continuación encontrarán el índice con enlaces a cada sección.

>[Deep Learning Avanzado (INF3813) - MIA.  Práctico Transferencia de conocimiento: Feature Extraction y Fine-tuning](#scrollTo=tHopPtVaNF1K)

>[Agenda](#scrollTo=kEloa5uXLIPK)

>[Actividad: Detección de Flores](#scrollTo=HOa9JfE2WfUb)

>>[Preámbulo](#scrollTo=CDBGNG3R4I8r)

>>[Primeros Pasos](#scrollTo=7AJawpaKoGdK)

>>>[Dataset](#scrollTo=OZDH2UsI89UR)

>>>[Entorno de entrenamiento](#scrollTo=boo4HVMclBbP)

>>[Modelo Base](#scrollTo=-QFHlOSR9BYd)

>>[Feature Extraction](#scrollTo=UQhGTF-8IB71)

>>[Finetuning](#scrollTo=131YF5zEX4_R)

>>[Resultados](#scrollTo=c903b2f4)

>>[¿Por qué funcionan las características?](#scrollTo=pqfEmbWhy01x)

>>>[Visualización del espacio de embeddings](#scrollTo=pqfEmbVis02x)

>>>[Búsqueda semántica](#scrollTo=pqfEmbSem03x)



# Actividad: Detección de Flores

En esta actividad aplicaremos lo que vimos en clase para un problema de clasificación de imágenes. Veremos como aplicar feature extraction y usar fine-tuning para este dominio en particular.

## Preámbulo

Primero debemos copiar el dataset hacia colab para luego decomprimirlo dentro del directorio `data`, que recién creamos. También importamos algunas de las librerías que ocuparemos durante esta actividad.

In [ ]:

import os

!mkdir -p data

# Descargamos nuestros datos para ser utilizados en este notebook.
if not os.path.exists('data/jigsaw-toxic-comment-classification-challenge.zip'):
    !gdown --id 1NjcTV1SeZgYWOAZG-ig2NBpYP5EyTy8Z -O data/jigsaw-toxic-comment-classification-challenge.zip
if not os.path.exists('data/flowers.zip'):
    !gdown --id 1ye0M-KnVtUbpIgGjogIjgWx0vReCthWA -O data/flowers.zip

!unzip -nq data/\*.zip -d data

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Fijemos seed para mantener la reproducibilidad de esta actividad
random.seed(1666)
rng_state = np.random.get_state()
np.random.seed(1666)
torch.manual_seed(1666)
torch.cuda.manual_seed(1666)

## Primeros Pasos

En la gran mayoría de aplicaciones con las que se encontraran en el mundo real, difícilmente podrán tener una cantidad de **datos etiquetados** suficientes para entrenar un red neuronal profunda como las que han visto hasta ahora.

En esta actividad práctica veremos como enfrentarnos a esta situación utilizando las herramientas que vimos en clase y la librería PyTorch

### Dataset

Trabajaremos con el dataset **flowers**, el cual ya utilizaron anteriormente. Este dataset está compuesto por imágenes de flores centradas. Las imágenes son de tamaño variable. Y la tarea a realizar es la de clasificación de la especie que aparece en cada imágen.

Si ejecutaron el código anterior, ya tenemos nuestro dataset en el directorio `data`.

Luego, debemos crear un objeto de la clase `Dataset`, para eso nos ayudaremos con la clase `torchvision.datasets.ImageFolder`, que hereda de esta, y utilizaremos durante esta actividad. En este caso ya tenemos divididos los distintos sets que utilizaremos (`train`, `val` y `test`), cada uno en su propio directorio y además agregamos un directorio `all`, que contiene la totalidad de los datos que también utilizaremos.

In [ ]:
from torchvision.datasets import ImageFolder

root_dir = 'data/flowers'
flowers_dataset = ImageFolder(f'{root_dir}/all/')
train_dataset = ImageFolder(f'{root_dir}/train/')
val_dataset = ImageFolder(f'{root_dir}/val/')
test_dataset = ImageFolder(f'{root_dir}/test/')

Exploremos los datos para que podamos ver algunas de las características que estos tienen. Primero veamos con cuantos datos contamos.

In [ ]:
for split, ds in [('totales', flowers_dataset), ('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    print('{:<18s}{:>6,d}'.format(f'Imágenes {split}:', len(ds)))

Podemos ver algunas de las clases, aleatoriamente elegidas, en que están etiquetadas nuestras imágenes. Además veamos como se ven los datos, elijamos ejemplos aleatorios con su clase y mostremoslos.

**Insight:** Siempre es importante revisar manualmente los datos. Así nos hacemos una mejor idea de las características del dataset y podemos sacar *insights* que nos ayuden a mejorar el rendimiento de los modelos que vayamos a entrenar.

In [ ]:
for label in random.choices(flowers_dataset.classes, k=10):
    print(label)

In [ ]:
ridx = random.randint(0, len(flowers_dataset))
sample_img, sample_class = flowers_dataset[ridx]

w, h = sample_img.size
class_ = flowers_dataset.classes[sample_class]

print(f'Tamaño imagen: {w}x{h}')
print(f'Etiqueta:      {class_}')
sample_img

### Entorno de entrenamiento

Partiremos inicializando los distintos hiperparámetros que utilizaremos durante este práctico.

In [ ]:
batch_size = 128 #@param {type:"number"}
learning_rate = 5E-4 #@param {type:"number"}
n_epochs = 5 # @param {type:"slider", min:1, max:20, step:1}

In [ ]:
criterion = nn.CrossEntropyLoss()
n_classes = len(flowers_dataset.classes)

Creemos los `DataLoader`s, que nos permitiran armar batches para el entrenamiento. Además nos ayudarán entregandonos los datos de forma desordenada o aleatoria, como habrán visto en clases anteriores, esto siempre es recomendable pero solo es necesario para el set de entrenamiento.

In [ ]:
from torch.utils.data import DataLoader

loaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True), # shuffle = aleatorio
    'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
    'test': DataLoader(test_dataset, batch_size=batch_size, shuffle=False),
}

Para ahorrarnos trabajo, utilizaremos transformaciones. En primera instancia, solo nos preocuparemos de que todas las imágenes sean del mismo tamaño y están normalizadas. Además nos ayudaremos transformando automáticamente la imagen a un `torch.Tensor`, el cual podremos utilizar para el entrenamiento.

In [ ]:
from torchvision import transforms

# Pixel medio y std de Imagenet
mean_pixel = [0.485, 0.456, 0.406]
std_pixel = [0.229, 0.224, 0.225]

to_normalized_tensor = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=mean_pixel, std=std_pixel),
            ])
train_dataset.transform = transforms.Compose([
                transforms.Resize([224, 224]),
                to_normalized_tensor
            ])
val_dataset.transform = transforms.Compose([
                transforms.Resize([224, 224]),
                to_normalized_tensor
            ])
test_dataset.transform = transforms.Compose([
                transforms.Resize([224, 224]),
                to_normalized_tensor
            ])

Además, crearemos varias funciones que nos ayudarán el entrenaminto de modelos reiteradas veces en este práctico. Les recomendamos que traten de entender cada una de estas funciones pues podrán reforzar conocimientos que ya han aprendido en el diplomado hasta ahora, pero para efectos de esta actividad no es necesario hacerlo.

In [ ]:
import torchvision.models as models

def get_model(n_classes, pretrained=None, frozen=False):
    model = models.resnet18(pretrained=pretrained)
    if frozen:
        for param in model.parameters():
            param.requires_grad = False

    model.fc = nn.Linear(in_features=512, out_features=n_classes, bias=True)
    model.to(device) # Envíamos el modelo a la GPU si está disponible
    return model

def run_epoch(phase, model, loader, criterion, optimizer=None):
    n_batches = len(loader)
    if phase == 'train':
        model.train()
    elif phase == 'val' or 'test':
        model.eval()

    cum_loss = 0.0
    cum_acc = 0.0
    for n_batch, (x, y) in enumerate(loader, start=1):
        if phase == 'train': # Limpiamos los gradientes solo al entrenar
            optimizer.zero_grad()

        x = x.to(device)
        y = y.to(device)
        result = model(x) # Aquí se ejecuta nuestro modelo

        loss = criterion(result, y) # Calculamos la pérdida
        cum_loss += loss.item()
        _, preds = torch.max(result, 1) # Nuestra predicción es la salida de mayor valor entre las 10 posibles
        cum_acc += torch.sum(preds == y.data)
        if phase == 'train':
            loss.backward() # Hacemos backpropagation solo en train
            optimizer.step() # Actualizamos parámetros solo en train

        current_cum_loss = cum_loss / n_batch
        current_cum_acc = 100 * cum_acc / (n_batch * batch_size)
        print(f'\r{phase.upper()}-Batch {n_batch}/{n_batches} '
              f'Loss: {current_cum_loss:.4f} Acc: {current_cum_acc:.2f}%', end='')

    epoch_loss = float(cum_loss / n_batch)
    epoch_acc = float(100 * cum_acc / (n_batch * batch_size))
    return epoch_loss, epoch_acc

def run_training(model, loaders, optimizer, criterion, n_epochs):
    phases = ['train', 'val']
    history = {
        'train': {'loss': [], 'acc': []},
        'val': {'loss': [], 'acc': []}
    }

    print(f"Validating model before training")
    val_loss, val_acc = run_epoch('val', model, loaders['val'], criterion)
    print()
    history['val']['loss'].append(val_loss)
    history['val']['acc'].append(val_acc)
    for epoch in range(1, n_epochs + 1):
        print(f"Epoch N°{epoch}")
        for phase in phases:
            epoch_loss, epoch_acc = run_epoch(phase, model, loaders[phase], criterion, optimizer=optimizer)

            # Registramos las pérdidas y accuracy por época, por fase
            history[phase]['loss'].append(epoch_loss)
            history[phase]['acc'].append(epoch_acc)
            print("")

    return history

def run_testing(model, loader, criterion, n_epochs):
    test_loss, test_acc = run_epoch('test', model, loader, criterion)
    print(f'\n\nTest loss: {test_loss:.4f}, Test acc.: {test_acc:.2f}\n')
    return {'loss': test_loss, 'acc': test_acc}

Definamos también algunas funciones que nos ayudarán a visualizar el entrenamiento y sus resultados de mejor forma.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image

def graph_metrics(metricas_test, metricas_train, title=''):
    fig, ax = plt.subplots()
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%0.3f'))
    plt.plot(range(1, int(len(metricas_train)+1)), metricas_train, marker='o')
    plt.plot(range(int(len(metricas_test))), metricas_test, marker='o')
    plt.legend(['Train', 'Val'])
    plt.title(title)
    plt.show()

def graph_training_metrics(history):
    for metric in ['loss', 'acc']:
        title = f'{metric.capitalize()} vs N° Épocas'
        graph_metrics(history['val'][metric], history['train'][metric], title=title)

def print_report(train_history, test_performance):
    graph_training_metrics(train_history)

    train_loss = train_history['train']['loss'][-1]
    train_acc = train_history['train']['acc'][-1]
    val_loss = train_history['val']['loss'][-1]
    val_acc = train_history['val']['acc'][-1]
    test_loss = test_performance['loss']
    test_acc = test_performance['acc']
    print(f'Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%')
    print(f'Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}%')

def bold(text):
  return '\033[1m' + text + '\033[0m'

def show_prediction(model, dataset, sample_idx=None, topk=5):
    if sample_idx is None:
        sample_idx = random.randrange(len(dataset))

    image_path, label = dataset.samples[sample_idx]
    label = flowers_dataset.classes[label]

    x, y = dataset[sample_idx]
    x = x.to(device)
    x = x.unsqueeze(dim=0)
    probs = nn.functional.softmax(model(x), dim=1)
    topk_probs, topk_classes = torch.topk(probs, k=topk)
    print(bold('Ground Truth'))
    print(label, end='\n\n')

    print(bold(f'Top-{topk} Predictions'))
    for prob, label_idx in zip(topk_probs.squeeze(), topk_classes.squeeze()):
        label_idx, prob = int(label_idx.detach()), float(prob.detach())
        print('{:<19} (p={:.4f})'.format(flowers_dataset.classes[label_idx], prob))

    print()
    return Image.open(image_path)

## Modelo Base

Habiendo explorados los datos, prosigamos a entrenar un modelo base. Este modelo nos servirá para poder tener un punto de comparación, y así tener una mejor idea de cuanto ayuda usar las técnicas que vimos en clases. Entrenaremos un modelo típico para esta tarea, una [ResNet18](https://arxiv.org/abs/1512.03385) que es un modelo simple y liviano para mostrar. De todas maneras una mejora simple y directa que se puede hacer es utilizar una *ResNet* de mayor número de capas, i.e. 34, 50, 101 o 152, lo cual debería mejorar el rendimiento de clasificador.

Utilizamos las funciones que definimos arriba para crear, entrenar nuestro primer modelo y ver su rendimiento.

In [ ]:
from torch.optim import Adam

model = get_model(n_classes=n_classes)
optimizer = Adam(model.parameters(), lr=learning_rate)

base_history = run_training(model, loaders, optimizer, criterion, n_epochs)

También procedemos a ver su rendimiento final en el set de test.

In [ ]:
base_test_performance = run_testing(model, loaders['test'], criterion, n_epochs)

print_report(base_history, base_test_performance)

Inspeccionemos la predicción del modelo para un ejemplo.

In [ ]:
show_prediction(model, test_dataset, sample_idx=166)  # Si omiten el sample_idx este será elegido aleatoriamente

## Feature Extraction

Luego continuaremos explorando los métodos expuestos durante la clase, en esta caso aplicaremos transfer learning.

En esta actividad aprovecharemos un modelo que previamente fue entrenado en el dataset [Imagenet](http://www.image-net.org/), el cual se ha vuelto prácticamente un standard en el área cuando se tiene en frente una tarea en imágenes. Utilizaremos el mismo modelo, ResNet18, que antes para mantener la consistencia y nuestros resultados sean comparables.

Para este caso, es importante notar que utilizaremos el parámetro `frozen` de `get_model` seteado en `True` de manera de que no entrenemos el modelo completo y solo lo hagamos para la última capa.

In [ ]:
from torch.optim import Adam

model = get_model(n_classes=n_classes, pretrained=True, frozen=True)
optimizer = Adam(model.parameters(), lr=learning_rate)

dtl_history = run_training(model, loaders, optimizer, criterion, n_epochs)

También procedemos a ver su rendimiento final en el set de test.

In [ ]:
dtl_test_performance = run_testing(model, loaders['test'], criterion, n_epochs)

print_report(dtl_history, dtl_test_performance)

Inspeccionemos la predicción del modelo para un ejemplo.

In [ ]:
show_prediction(model, test_dataset, sample_idx=166)  # Si omiten el sample_idx este será elegido aleatoriamente

## Finetuning

Entrenemos un modelo haciendo finetuning de uno originalmente entrenado en Imagenet. Para esto es necesario conseguirnos los pesos de dicho modelo, por suerte PyTorch nos hace la vida fácil y con un simple parámetro podemos descargarlos automáticamente.

Para aprovechar los modelos de PyTorch debemos utilizar los modelos que vienen en [`torchvision.models`](https://pytorch.org/docs/stable/torchvision/models.html). Debemos crear un modelo llamando seteando el parámetro `pretrained=True`, esto automáticamente descargará (fijense en el *output* al momento de empezar crear el modelo) los pesos y los usará para el modelo.

Es importante notar que la mayoría de estos modelos son preentrandos en Imagenet, pero hay algunos con otros datasets de origen, por lo que se recomienda revisar la documentación previamente a usar alguno de los modelos disponibles.

In [ ]:
ft_model = models.resnet18(pretrained=True)
ft_model.fc = nn.Linear(in_features=512, out_features=n_classes, bias=True)
ft_model.to(device)
None # No queremos el output de la descripción del modelo en colab

Ahora hagamos el finetuning de nuestro modelo.

In [ ]:
learning_rate = 5e-4 # podríamos usar 0.01 la learning rate 1/10 de la original (para ResNet 0.1)

ft_model = get_model(n_classes=n_classes, pretrained=True, frozen=False)
ft_optimizer = Adam(ft_model.parameters(), lr=learning_rate)

ft_history = run_training(ft_model, loaders, ft_optimizer, criterion, n_epochs)
ft_test_performance = run_testing(ft_model, loaders['test'], criterion, n_epochs)

print_report(ft_history, ft_test_performance)

Inspeccionemos la predicción del modelo para un ejemplo.

In [ ]:
show_prediction(ft_model, test_dataset, sample_idx=166) # Si omiten el sample_idx este será elegido aleatoriamente

In [ ]:
ft_model = ft_optimizer = aug_model = aug_optimizer = model = optimizer = criterion = None
torch.cuda.empty_cache()

## Resultados

A continuación, presentamos una comparación del rendimiento de los tres modelos entrenados: el modelo base, el modelo con Feature Extraction y el modelo con Fine-tuning. Se mostrarán tanto los resultados en formato de tabla como en gráficos para visualizar las métricas de entrenamiento, validación y prueba.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns # Import seaborn for nicer colors and styles

def plot_comparison_metric(ax, metric_name, base, dtl, ft, base_test, dtl_test, ft_test):
    labels = ['Base Model', 'Feature Extraction', 'Finetuning']
    train_values = [base['train'][metric_name][-1], dtl['train'][metric_name][-1], ft['train'][metric_name][-1]]
    val_values = [base['val'][metric_name][-1], dtl['val'][metric_name][-1], ft['val'][metric_name][-1]]
    test_values = [base_test[metric_name], dtl_test[metric_name], ft_test[metric_name]]

    x = range(len(labels))
    width = 0.2

    # Use seaborn style and palette
    sns.set_style("whitegrid")
    colors = sns.color_palette("viridis", 3) # Choose a nice color palette

    rects1 = ax.bar([i - width for i in x], train_values, width, label='Train', color=colors[0])
    rects2 = ax.bar(x, val_values, width, label='Validation', color=colors[1])
    rects3 = ax.bar([i + width for i in x], test_values, width, label='Test', color=colors[2])

    ax.set_ylabel(metric_name.capitalize())
    ax.set_title(f'Comparison of {metric_name.capitalize()} across Models', fontsize=10) # Adjust title font size
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8) # Adjust x-axis label font size
    ax.legend(fontsize=8) # Adjust legend font size

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.2f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=7) # Adjust label font size

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)

In [ ]:
print("--- Model Performance Comparison ---\n")
print(f"{'Model':<30s}{'Train Acc (%)':<20s}{'Val Acc (%)':<20s}{'Test Acc (%)':<20s}")
print("-" * 90)
print(f"{'Base Model':<30s}{base_history['train']['acc'][-1]:<20.2f}{base_history['val']['acc'][-1]:<20.2f}{base_test_performance['acc']:<20.2f}")
print(f"{'Feature Extraction':<30s}{dtl_history['train']['acc'][-1]:<20.2f}{dtl_history['val']['acc'][-1]:<20.2f}{dtl_test_performance['acc']:<20.2f}")
print(f"{'Finetuning':<30s}{ft_history['train']['acc'][-1]:<20.2f}{ft_history['val']['acc'][-1]:<20.2f}{ft_test_performance['acc']:<20.2f}")

print("\n")
print(f"{'Model':<30s}{'Train Loss':<20s}{'Val Loss':<20s}{'Test Loss':<20s}")
print("-" * 90)
print(f"{'Base Model':<30s}{base_history['train']['loss'][-1]:<20.4f}{base_history['val']['loss'][-1]:<20.4f}{base_test_performance['loss']:<20.4f}")
print(f"{'Feature Extraction':<30s}{dtl_history['train']['loss'][-1]:<20.4f}{dtl_history['val']['loss'][-1]:<20.4f}{dtl_test_performance['loss']:<20.4f}")
print(f"{'Finetuning':<30s}{ft_history['train']['loss'][-1]:<20.4f}{ft_history['val']['loss'][-1]:<20.4f}{ft_test_performance['loss']:<20.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4)) # Create a figure with two subplots side-by-side

plot_comparison_metric(ax1, 'acc', base_history, dtl_history, ft_history, base_test_performance, dtl_test_performance, ft_test_performance)
plot_comparison_metric(ax2, 'loss', base_history, dtl_history, ft_history, base_test_performance, dtl_test_performance, ft_test_performance)

fig.tight_layout()
plt.show()

## ¿Por qué funcionan las características?

Como vimos en clase, una red preentrenada transforma cada imagen en un **vector de características** (*embedding*): un vector $n$-dimensional que la representa. En ResNet18 este vector es la salida de la capa de *average pooling*, justo antes del cabezal de clasificación (`fc`), y tiene 512 dimensiones. Es exactamente lo que aprovechamos en Feature Extraction: congelamos la red y solo entrenamos un clasificador sobre estos vectores.

Para entender por qué esto funciona tan bien, compararemos las representaciones de dos redes con la misma arquitectura, **ninguna de las cuales ha sido entrenada con flores**:

- Una ResNet18 con **pesos aleatorios**.
- Una ResNet18 **preentrenada en ImageNet**.

Si el preentrenamiento aporta conocimiento útil, las imágenes de una misma especie deberían quedar cerca entre sí en el espacio de embeddings, aunque la red nunca haya visto estas clases.

Partimos definiendo un extractor de características: tomamos el modelo y reemplazamos su cabezal por la identidad (`nn.Identity`), de modo que la salida sea directamente el embedding.

In [ ]:
def get_feature_extractor(pretrained=None):
    # Reutilizamos get_model y reemplazamos el cabezal por la identidad,
    # así la red entrega directamente el vector de 512 dimensiones.
    extractor = get_model(n_classes=n_classes, pretrained=pretrained)
    extractor.fc = nn.Identity()
    extractor.eval()
    return extractor

@torch.no_grad()  # Solo hacemos inferencia, no necesitamos gradientes
def compute_embeddings(extractor, dataset):
    # Sin shuffle, para que el i-ésimo embedding corresponda a dataset.samples[i]
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    embeddings, labels = [], []
    for n_batch, (x, y) in enumerate(loader, start=1):
        embeddings.append(extractor(x.to(device)).cpu())
        labels.append(y)
        print(f'\rBatch {n_batch}/{len(loader)}', end='')
    print()
    return torch.cat(embeddings), torch.cat(labels)

Calculamos los embeddings de los sets de train y test con ambas redes. Esto puede tomar un par de minutos.

In [ ]:
torch.manual_seed(1666)
extractors = {
    'Pesos aleatorios': get_feature_extractor(pretrained=None),
    'Preentrenada en ImageNet': get_feature_extractor(pretrained=True),
}

embeddings = {}
for name, extractor in extractors.items():
    print(bold(name))
    train_emb, train_labels = compute_embeddings(extractor, train_dataset)
    test_emb, test_labels = compute_embeddings(extractor, test_dataset)
    embeddings[name] = {'train': train_emb, 'test': test_emb}

extractors = None  # Ya no necesitamos las redes, liberamos memoria
torch.cuda.empty_cache()

print(f'\nEmbeddings de train: {tuple(train_emb.shape)}')
print(f'Embeddings de test:  {tuple(test_emb.shape)}')

### Visualización del espacio de embeddings

No podemos graficar vectores de 512 dimensiones directamente, así que los proyectamos a 2 dimensiones con [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html), una técnica que intenta preservar qué puntos están cerca de cuáles. Para que el gráfico sea legible, mostramos solo 10 especies elegidas al azar.

**Importante:** ninguna de las dos redes fue entrenada para clasificar flores. Las etiquetas solo se usan para colorear los puntos.

In [ ]:
from sklearn.manifold import TSNE

# Elegimos 10 especies al azar para que el gráfico sea legible
rng = np.random.default_rng(1666)
selected_classes = rng.choice(n_classes, size=10, replace=False)
mask = np.isin(train_labels.numpy(), selected_classes)
subset_labels = train_labels.numpy()[mask]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (name, emb) in zip(axes, embeddings.items()):
    points = TSNE(n_components=2, perplexity=30, init='pca',
                  random_state=1666).fit_transform(emb['train'][mask].numpy())
    for c in selected_classes:
        idx = subset_labels == c
        ax.scatter(points[idx, 0], points[idx, 1], s=14, alpha=0.8,
                   label=train_dataset.classes[c])
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])

axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
fig.suptitle('Embeddings de ResNet18 proyectados con t-SNE (train, 10 especies)')
fig.tight_layout()
plt.show()

El gráfico es cualitativo. Para medir qué tan buena es cada representación usaremos un clasificador **k-NN** (*k vecinos más cercanos*): cada imagen de test recibe la especie más frecuente entre sus 5 vecinos más cercanos de train. Este clasificador no tiene parámetros que entrenar, por lo que su resultado depende solo de la calidad de los embeddings.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print(f"{'Representación':<30s}{'k-NN Test Acc (%)':<20s}")
print('-' * 50)
for name, emb in embeddings.items():
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(emb['train'].numpy(), train_labels.numpy())
    knn_acc = 100 * knn.score(emb['test'].numpy(), test_labels.numpy())
    print(f'{name:<30s}{knn_acc:<20.2f}')

**Insight:** con pesos aleatorios las especies aparecen mezcladas, mientras que con la red preentrenada forman grupos bien definidos, aunque la red nunca vio estas clases. El preentrenamiento en un dataset masivo y diverso le enseñó a extraer características generales (colores, texturas, formas, partes) que también sirven para distinguir flores. Esto es lo que explica el buen rendimiento de Feature Extraction: un clasificador simple sobre estos vectores ya separa bien las clases. Comparen el accuracy de k-NN con el obtenido en la sección de Feature Extraction.

**Para explorar:** repitan la visualización con el modelo de fine-tuning (para esto, no lo liberen en la celda que limpia la memoria). ¿Cómo cambian los grupos cuando las representaciones se adaptan a la tarea de destino?

### Búsqueda semántica

Una aplicación directa de los embeddings es la **búsqueda semántica**: en lugar de buscar por palabras clave o metadatos, buscamos por *similitud de contenido*. La idea es simple:

1. Calculamos el embedding de todas las imágenes de nuestra "base de datos" (en este caso, el set de train).
2. Calculamos el embedding de la imagen de consulta (una imagen de test).
3. Retornamos las imágenes cuyos embeddings son más similares a la consulta, usando la **similitud coseno**.

Esto es lo que hacen a gran escala las bases de datos vectoriales que vimos en clase, que además usan búsqueda aproximada de vecinos cercanos para escalar a millones de vectores.

In [ ]:
import torch.nn.functional as F

def semantic_search(query_emb, database_emb, k=5):
    # Al normalizar los vectores, el producto punto equivale a la similitud coseno
    query = F.normalize(query_emb.unsqueeze(0), dim=1)
    database = F.normalize(database_emb, dim=1)
    similarities = (query @ database.T).squeeze(0)
    top_sims, top_idx = torch.topk(similarities, k=k)
    return top_idx.tolist(), top_sims.tolist()

def load_image(path, size=224):
    return Image.open(path).convert('RGB').resize((size, size))

def show_search_results(query_idx=None, k=5):
    if query_idx is None:
        query_idx = random.randrange(len(test_dataset))
    query_path, query_label = test_dataset.samples[query_idx]
    query_class = test_dataset.classes[query_label]

    fig, axes = plt.subplots(len(embeddings), k + 1, figsize=(2.5 * (k + 1), 3.2 * len(embeddings)))
    for row, (name, emb) in enumerate(embeddings.items()):
        idxs, sims = semantic_search(emb['test'][query_idx], emb['train'], k=k)

        axes[row, 0].imshow(load_image(query_path))
        axes[row, 0].set_title(f'CONSULTA\n{query_class}', fontsize=9, fontweight='bold')
        axes[row, 0].set_ylabel(name, fontsize=10, fontweight='bold')

        for col, (i, sim) in enumerate(zip(idxs, sims), start=1):
            path, label = train_dataset.samples[i]
            color = 'green' if label == query_label else 'red'  # verde = misma especie
            axes[row, col].imshow(load_image(path))
            axes[row, col].set_title(f'{train_dataset.classes[label]}\nsim={sim:.3f}',
                                     fontsize=9, color=color)

    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    fig.suptitle(f'Top-{k} imágenes de train más similares a la consulta', fontsize=12)
    fig.tight_layout()
    plt.show()

show_search_results(query_idx=166)  # Misma imagen de test que usamos en las secciones anteriores

Prueben con otras consultas elegidas al azar ejecutando la siguiente celda varias veces. Los títulos en verde indican que el resultado es de la misma especie que la consulta.

In [ ]:
show_search_results()  # Si omiten el query_idx este será elegido aleatoriamente

**Insight:** con la red preentrenada, los resultados suelen ser de la misma especie y muy parecidos visualmente a la consulta, mientras que con pesos aleatorios los vecinos tienen poco que ver con ella (a lo más comparten colores generales). Fíjense también en los valores de similitud: en la red aleatoria suelen ser altos y muy parecidos entre sí, lo que indica que esos vectores no distinguen bien entre imágenes.

Aquí la consulta es una imagen. Para buscar con texto (por ejemplo, *"flores amarillas con pétalos largos"*) necesitamos un modelo que ubique texto e imágenes en un mismo espacio de embeddings, como CLIP. Es la misma idea de búsqueda semántica que vimos en clase, extendida a varias modalidades.